# Reconstruct TEM Subfigures from Source Index

This notebook crops TEM subfigures from original figures using the coordinates provided in `TEM_source_index.csv`.

## Prerequisites

Before running this notebook, download the original figures from their source articles.
Each figure page URL can be constructed as:

```
{ARTICLE_URL}/figures/{N}
```

where `{N}` is the figure number extracted from `FIG_ID` (e.g., `figure-3` → `3`).

Save each downloaded figure as `{ARTICLE_ID}_{FIG_ID}.jpg` and place it in `FIGURE_DIR`.

## Import Packages

In [ ]:
import os
import pandas as pd
from PIL import Image

## Settings

Adjust the paths below to match your local setup:

- `SOURCE_INDEX`: path to `TEM_source_index.csv`
- `FIGURE_DIR`: directory containing your downloaded original figures
- `OUTPUT_DIR`: directory where cropped subfigures will be saved

In [ ]:
SOURCE_INDEX = "TEM_source_index.csv"
FIGURE_DIR   = "Data/clean_raw_figures"
OUTPUT_DIR   = "Data/TEM_subfigures"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load Source Index

In [ ]:
df = pd.read_csv(SOURCE_INDEX)
print(f"Source index loaded: {len(df)} subfigures")

## Crop Subfigures

For each `(ARTICLE_ID, FIG_ID)` group, the notebook:
1. Locates the corresponding original figure in `FIGURE_DIR`
2. Converts normalized bounding box coordinates to pixel coordinates
3. Crops and saves each subfigure to `OUTPUT_DIR`

In [ ]:
done, missing = 0, []

for (article_id, fig_id), group in df.groupby(["ARTICLE_ID", "FIG_ID"]):
    fig_name = f"{article_id}_{fig_id}.jpg"
    fig_path = os.path.join(FIGURE_DIR, fig_name)

    if not os.path.exists(fig_path):
        missing.append(fig_name)
        continue

    img = Image.open(fig_path).convert("RGB")
    W, H = img.size

    for _, row in group.iterrows():
        xc, yc = row["X_CENTER"], row["Y_CENTER"]
        bw, bh = row["WIDTH"], row["HEIGHT"]

        # Convert normalized coordinates to pixel coordinates
        x1 = int((xc - bw / 2) * W)
        y1 = int((yc - bh / 2) * H)
        x2 = int((xc + bw / 2) * W)
        y2 = int((yc + bh / 2) * H)

        # Clamp to image boundaries
        x1 = max(0, min(x1, W - 1))
        y1 = max(0, min(y1, H - 1))
        x2 = max(0, min(x2, W))
        y2 = max(0, min(y2, H))

        if x2 <= x1 or y2 <= y1:
            continue

        crop = img.crop((x1, y1, x2, y2)).convert("RGB")
        out_path = os.path.join(OUTPUT_DIR, row["TEM_SUB_IMAGE_ID"])
        crop.save(out_path, format="PNG")
        done += 1


## Results

In [ ]:
print(f"Done: {done} subfigures saved to {OUTPUT_DIR}")

if missing:
    print(f"\nMissing {len(missing)} original figures (not found in {FIGURE_DIR}):")
    for m in missing[:20]:
        print("  ", m)
    if len(missing) > 20:
        print(f"  ... and {len(missing) - 20} more")
else:
    print("All original figures found.")